# Análisis descriptivo del conjunto de datos

Este notebook realiza un análisis descriptivo del conjunto de datos `dataset_train.csv`, incluyendo visualizaciones gráficas para explorar las variables.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
# Librerias de manipulación de datos
from pyspark.sql import functions as F

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Cargar el conjunto de datos

Se cargará la tabla `silver_table` para el análisis.

In [0]:
# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

SILVER_FULL = qname(silver_table)
print("Tabla Silver:", SILVER_FULL)

In [0]:

# Leer datos de la tabla Bronze
lpn_silver = spark.table(SILVER_FULL)

display(lpn_silver.limit(20))

In [0]:
# Dimensiones del dataset
print(f"Filas: {lpn_silver.count()}, Columnas: {len(lpn_silver.columns)}")

In [0]:
lpn_silver.printSchema()

## 3. Resumen estadístico de las variables

Se presentan estadísticas descriptivas de las variables numéricas y frecuencias de las variables categóricas.

In [0]:
# Nulos por columna
nulls_by_col = lpn_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in lpn_silver.columns]).collect()[0].asDict()
print("🧯 Nulos por columna:")
for k, v in sorted(nulls_by_col.items(), key=lambda kv: kv[1], reverse=True):
    if v > 0:
        print(f"- {k}: {v:,}")

In [0]:
# Identificar columnas numéricas para realizar análisis estadístico
dtypes = dict(lpn_silver.dtypes)
numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}
num_cols = [c for c,t in dtypes.items() if any(t.startswith(nt) for nt in numeric_types)]

print(f"📊 Columnas numéricas ({len(num_cols)}): {num_cols}")

if num_cols:
    display(lpn_silver.select(num_cols).summary("count","mean","stddev","min","25%","50%","75%","max"))
else:
    print("No se detectaron columnas numéricas.")


In [0]:
# Calcular asimetría (skew) de variables numéricas
skewness = lpn_silver[num_cols].toPandas().skew().sort_values(ascending=False)
print("Asimetría (skew) de variables numéricas:")
display(skewness)


In [0]:
# Cardinalidad por columna (rápido para string/boolean; cuidado con altas cardinalidades)
string_like_cols = [c for c,t in dtypes.items() if t in ("string", "boolean")]
print(f"📊 Columnas string/bool ({len(string_like_cols)}): {string_like_cols}")
card = []
for c in string_like_cols:
    try:
        card.append((c, lpn_silver.select(c).distinct().count()))
    except:
        card.append((c, None))
print("🔠 Cardinalidad de columnas string/bool:")
for c, k in sorted(card, key=lambda x: (x[1] if x[1] is not None else -1), reverse=True):
    print(f"- {c}: {k}")



En base a la cardinalidad se puede apreciar que la variable CASE_NBR es el id del LPN por eso todos sus valores son diferentes

## 4. Gráficos de distribución de variables numéricas

Se visualizan histogramas y boxplots para analizar la distribución de las variables numéricas.

In [0]:
# convertir a dataframe pandas
df_train_pd = lpn_silver.toPandas()

In [0]:
# Histogramas y boxplots para variables numéricas

for col in num_cols:
    # Eliminar valores nulos para evitar errores en boxplot
    data = df_train_pd[col].dropna()
    if len(data) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(data, ax=axes[0], kde=True, color='skyblue')
        axes[0].set_title(f'Histograma de {col} (Train)')
        sns.boxplot(x=data, ax=axes[1], color='lightgreen')
        axes[1].set_title(f'Boxplot de {col} (Train)')
        plt.tight_layout()
        plt.show()
    else:
        print(f'La columna {col} no tiene datos válidos para graficar.')

## Correlación entre variables numéricas

Se calcula la matriz de correlación y se visualiza mediante un heatmap.

In [0]:
# Matriz de correlación y heatmap
corr = df_train_pd[num_cols].corr(method="pearson")
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación (Train)')
plt.show()

In [0]:
# graficos para variables numericas vs target
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df_train_pd[col], y=df_train_pd['LABEL_ZONE'], palette='Set2')
    plt.title(col)
    plt.show()

## 5. Gráficos de variables categóricas

Se visualizan gráficos de barras para mostrar la frecuencia de las categorías.

In [0]:
# Gráficos de barras para variables categóricas

cat_cols = df_train_pd.select_dtypes(include='object').columns
cat_cols_greather13 = [col for col in cat_cols if df_train_pd[col].nunique() > 13]
print(f'Columnas categóricas con más de 13 categorías: {cat_cols_greather13} {len(cat_cols_greather13)}')
cat_cols = [col for col in cat_cols if df_train_pd[col].nunique() <= 13]  # Filtrar solo categorías con 13 o menos valores únicos
print(f'Columnas categóricas con 13 o menos categorías: {cat_cols} {len(cat_cols)}')

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    ax = sns.countplot(data=df_train_pd, x=col, hue=col, palette='Set2')
    plt.title(f'Frecuencia de categorías en {col} (Train)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Calcular porcentajes y agregar etiquetas solo si el porcentaje es mayor a 0
    total = len(df_train_pd)
    for p in ax.patches:
        height = p.get_height()
        percent = 100 * height / total
        if percent > 0:
            ax.annotate(f'{percent:.1f}%', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=9, color='black', rotation=0)
    
    plt.show()

In [0]:
# graficos para variables categoricas vs target
for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df_train_pd, x=col, hue='LABEL_ZONE', palette='Set2')
    plt.title(f'Frecuencia de categorías en {col} (Train)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Análisis de la Variable Objetivo

In [0]:
# Grafico de distribucion de variable target LABEL_ZONE
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df_train_pd, x="LABEL_ZONE", hue="LABEL_ZONE", palette='Set2')
plt.title(f'Frecuencia de categorías en LABEL_ZONE (Train)')
plt.xticks(rotation=45)
plt.tight_layout()

# Calcular porcentajes y agregar etiquetas solo si el porcentaje es mayor a 0
total = len(df_train_pd)
for p in ax.patches:
    height = p.get_height()
    percent = 100 * height / total
    if percent > 0:
        ax.annotate(f'{percent:.1f}%', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=9, color='black', rotation=0)

plt.show()